In [1]:
"""
Compare the growth factor limitation of each PFT

Plot the ratio of GPP/LAI against
* air temperature  (TBOT K) (h0)
* soil water content (H2OSOI, kg m-2) (h0) (did not save the liquid variable)
* incoming solar radiation (FSDS, W m-2) (h0)
* soil mineral nitrogen content (gN m-2: SMINN, SMIN_NO3, SMIN_NH4) (h0)
* soil phosphorus content (gP m-2: SMINP, SOLUTIONP) (h0)

Using daily values
"""
import numpy as np
import os
import pandas as pd
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt
import itertools as it
from scipy.stats import linregress
from matplotlib.cm import get_cmap
from utils.constants import chamber_levels_complete

## Extract the data

In [2]:
def get_filelist(prefix, plot, stream):
    rundir = os.path.join(os.environ['PROJDIR'], 'E3SM', 'output', 
                          f'{prefix}_US-SPR_ICB20TRCNPRDCTCBC/spruce_treatments/plot{plot}_US-SPR_ICB20TRCNPRDCTCBC/run')
    flist = sorted(glob(os.path.join(rundir, f'plot{plot}*.{stream}.*.nc')))[:-1]
    return flist


def get_e3sm_grid(col_var, handle):
    # grid-level outputs

    LEVGRND = np.array([0.007100635, 0.027925, 0.06225858, 0.1188651, 0.2121934,
                        0.3660658, 0.6197585, 1.038027, 1.727635, 2.864607, 4.739157,
                        7.829766, 12.92532, 21.32647, 35.17762])
    LEVGRND_I = np.append(np.insert(
        (LEVGRND[1:] + LEVGRND[:-1])*0.5, 0, 0
    ), LEVGRND[-1] + 0.5 * (LEVGRND[-1] - LEVGRND[-2]))
    THICKNESS = np.diff(LEVGRND_I)

    if col_var == "TBOT":
        data = handle[col_var] - 273.15
        unit = "degC"
    elif col_var == "SM15":
        data = (
            handle["H2OSOI"][:, 0, :] * THICKNESS[0]
            + handle["H2OSOI"][:, 1, :] * THICKNESS[1]
            + handle["H2OSOI"][:, 2, :] * THICKNESS[2]
            + handle["H2OSOI"][:, 3, :] * THICKNESS[3]
            + handle["H2OSOI"][:, 4, :] * (0.15 - LEVGRND_I[4])
        ) / 0.15
        unit = "mm3/mm3"
    else:
        data = handle[col_var]
        unit = handle[col_var].attrs["units"]
    return data, unit


def get_e3sm_pft(pft, hol_add, pft_var, handle):
    data = handle[pft_var][:, pft + hol_add]
    unit = handle[pft_var].attrs["units"]
    return data, unit


# Read and save the variables
plot_list = ['04', '06', '07', '08', '10', '11', '13', '16', '17', '19', '20']
prefix = '20240304'
#prefix = '20231112'
outdir = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract', prefix)
if not os.path.exists(outdir):
    os.mkdir(outdir)

grid_var_list = ['TBOT', 'SM15', 'FSDS', 'SMINN', 'SMIN_NO3', 'SMIN_NH4', 
                 'SMINP', 'SOLUTIONP', 'GPP', 'TOTVEGC', 'FPG', 'FPG_P']
grid_unit_list = ['degC', 'mm3/mm3', 'W/m^2', 'gN/m^2', 'gN/m^2', 'gN/m^2',
                  'gP/m^2', 'gP/m^2', '', '']
pft_var_list = ['GPP', 'TLAI', 'TOTVEGC', 'NPP', 'BGNPP', 'MR']
pft_unit_list = ['gC/m^2/s', '1']

In [3]:
# Grid level variables
stream = 'h1'
collect = pd.DataFrame(np.nan, 
                       index = pd.date_range('2015-01-01', '2020-12-31', freq = '1D'),
                       columns = pd.MultiIndex.from_product([grid_var_list, plot_list, 
                                                             ['hum', 'hol', 'avg']]))
collect = collect.loc[~((collect.index.month == 2) & (collect.index.day == 29)), :]
for plot in plot_list:
    files = get_filelist(prefix, plot, stream)
    hr = xr.open_mfdataset(files)
    for var in grid_var_list:
        data, _ = get_e3sm_grid(var, hr)
        collect.loc[:, (var, plot, 'hum')] = data.values[:, 0]
        collect.loc[:, (var, plot, 'hol')] = data.values[:, 1]
        collect.loc[:, (var, plot, 'avg')] = data.values[:, 0] * 0.64 + data.values[:, 1] * 0.36
    hr.close()
collect.to_csv(os.path.join(outdir, 'factorial_limitation_grid.csv'))

In [4]:
# pft-level variables
hol_add = 17
pft_list = [2,3,11,12]
stream = 'h2'
collect = pd.DataFrame(np.nan, 
                       index = pd.date_range('2015-01-01', '2020-12-31', freq = '1D'),
                       columns = pd.MultiIndex.from_product([pft_var_list, pft_list, plot_list, 
                                                             ['hum', 'hol', 'avg']]))
collect = collect.loc[~((collect.index.month == 2) & (collect.index.day == 29)), :]
for plot in plot_list:
    files = get_filelist(prefix, plot, stream)
    hr = xr.open_mfdataset(files)
    for var in pft_var_list:
        for pft in pft_list:
            data, _ = get_e3sm_pft(pft, 0, var, hr)
            collect.loc[:, (var, pft, plot, 'hum')] = data.values
            data, unit = get_e3sm_pft(pft, hol_add, var, hr)
            collect.loc[:, (var, pft, plot, 'hol')] = data.values
            collect.loc[:, (var, pft, plot, 'avg')] = \
                collect.loc[:, (var, pft, plot, 'hum')] * 0.64 + \
                collect.loc[:, (var, pft, plot, 'hol')] * 0.36
    hr.close()
collect.to_csv(os.path.join(outdir, 'factorial_limitation_pft.csv'))

## Plot the relationships with temperature

In [6]:
# read data
grid_data = pd.read_csv(os.path.join(outdir, 'factorial_limitation_grid.csv'), 
                        index_col = 0, parse_dates = True, header = [0,1,2])
pft_data = pd.read_csv(os.path.join(outdir, 'factorial_limitation_pft.csv'), 
                       index_col = 0, parse_dates = True, header = [0,1,2,3])
product_ratio = pft_data.loc[:, 'GPP'] / pft_data.loc[:, 'TLAI']

# limit to high-growing-season (summer)
grid_data = grid_data.loc[grid_data.index.month.isin([6,7,8]), :]
product_ratio = product_ratio.loc[product_ratio.index.month.isin([6,7,8]), :]

In [7]:
loc = 'avg'
#for (var, unit), loc in it.product(zip(grid_var_list, grid_unit_list), ['hum', 'hol', 'avg']):
for var, unit in zip(grid_var_list, grid_unit_list):
    fig, axes = plt.subplots(2, 2, figsize = (10, 10), sharex = True, sharey = True)
    for i, pft in enumerate(pft_list):
        ax = axes.flat[i]

        x = grid_data.loc[:, (var, slice(None), loc)].values.reshape(-1)
        y = product_ratio.loc[:, (str(pft), slice(None), loc)].values.reshape(-1)
        ax.plot(x, y, 'ob')

        # Plot fitted linear regression line
        slope, intercept, r_value, p_value, std_err = linregress(x, y)

        x_fit = np.linspace(x.min(), x.max(), 20)
        y_fit = intercept + slope * x_fit

        ci = 1.96 * std_err
        y_upper = y_fit + ci
        y_lower = y_fit - ci

        ax.plot(x_fit, y_fit, color = 'k')
        #ax.plot(x_fit, y_lower, '--', color = colors2[j])
        #ax.plot(x_fit, y_upper, '--', color = colors2[j])

        if i >= 2:
            ax.set_xlabel(f'{var} ({unit})')
        if np.mod(i, 2) == 0:
            ax.set_ylabel('GPP / LAI (gC m-2 s-1)')
        ax.set_title(f'pft = {pft}')
    fig.savefig(os.path.join(outdir, f'factorial_limitation_{var}.png'), 
                dpi = 600., bbox_inches = 'tight')
    plt.close(fig)

In [15]:
# Compare N & P limitation of two simulations
outdir = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract')

default = pd.read_csv(os.path.join(outdir, '20231112', 'factorial_limitation_grid.csv'), 
                      index_col = 0, parse_dates = True, header = [0,1,2])
default = default.loc[default.index.month.isin([8]), :]
rootpheno = pd.read_csv(os.path.join(outdir, '20240304', 'factorial_limitation_grid.csv'), 
                      index_col = 0, parse_dates = True, header = [0,1,2])
rootpheno = rootpheno.loc[rootpheno.index.month.isin([8]), :]

default_pft = pd.read_csv(os.path.join(outdir, '20231112', 'factorial_limitation_pft.csv'), 
                      index_col = 0, parse_dates = True, header = [0,1,2,3])
default_pft = default_pft.loc[default_pft.index.month.isin([8]), :]
rootpheno_pft = pd.read_csv(os.path.join(outdir, '20240304', 'factorial_limitation_pft.csv'), 
                            index_col = 0, parse_dates = True, header = [0,1,2,3])
rootpheno_pft = rootpheno_pft.loc[rootpheno_pft.index.month.isin([8]), :]

In [16]:
loc = 'avg'
fig, axes = plt.subplots(1, 2, figsize = (10, 5), sharex = True, sharey = True)
for i, var in enumerate(['FPG', 'FPG_P']):
    ax = axes.flat[i]

    x = default.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
    y = default.loc[:, (var, slice(None), loc)].values.reshape(-1)
    ax.plot(x, y, 'o', label = 'Default', ms = 1)

    x = rootpheno.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
    y = rootpheno.loc[:, (var, slice(None), loc)].values.reshape(-1)
    ax.plot(x, y, 'o', label = 'RootPheno', ms = 1)
    ax.legend()
    ax.set_title(var)
fig.savefig(os.path.join(outdir, '..', f'factorial_limitation_compare.png'), dpi = 600., 
            bbox_inches = 'tight')
plt.close(fig)

In [17]:
loc = 'avg'
fig, axes = plt.subplots(2, 3, figsize = (13, 10), sharex = True, sharey = False)
for i, var in enumerate(['SM15', 'GPP', 'TOTVEGC', 'GPP/TOTVEGC', 'SMINN', 'SMINP']):
    ax = axes.flat[i]

    x = default.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
    if var == 'GPP/TOTVEGC':
        y = default.loc[:, ('GPP', slice(None), loc)].values.reshape(-1) / \
            default.loc[:, ('TOTVEGC', slice(None), loc)].values.reshape(-1)
    else:
        y = default.loc[:, (var, slice(None), loc)].values.reshape(-1)
    ax.plot(x, y, 'o', label = 'Default', ms = 1)

    x = rootpheno.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
    if var == 'GPP/TOTVEGC':
        y = rootpheno.loc[:, ('GPP', slice(None), loc)].values.reshape(-1) / \
            rootpheno.loc[:, ('TOTVEGC', slice(None), loc)].values.reshape(-1)
    else:
        y = rootpheno.loc[:, (var, slice(None), loc)].values.reshape(-1)

    ax.plot(x, y, 'o', label = 'RootPheno', ms = 1)
    ax.legend()
    ax.set_title(var)
fig.savefig(os.path.join(outdir, '..', f'factorial_limitation_compare2.png'), dpi = 600., 
            bbox_inches = 'tight')
plt.close(fig)

In [18]:
loc = 'avg'
for ratio in ['BGNPP/NPP','NPP/GPP','GPP/TOTVEGC', 'GPP/TLAI', 'NPP/GPP']:
    fig, axes = plt.subplots(2, 2, figsize = (10, 10), sharex = True, sharey = True)
    for i, pft in enumerate([2, 3, 11, 12]):
        ax = axes.flat[i]

        top = ratio.split('/')[0]
        base = ratio.split('/')[1]

        x = default.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
        y = default_pft.loc[:, (top, str(pft), slice(None), loc)].values.reshape(-1) / \
            default_pft.loc[:, (base, str(pft), slice(None), loc)].values.reshape(-1)
        ax.plot(x, y, 'o', label = 'Default', ms = 1)

        x = rootpheno.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
        y = rootpheno_pft.loc[:, (top, str(pft), slice(None), loc)].values.reshape(-1) / \
            rootpheno_pft.loc[:, (base, str(pft), slice(None), loc)].values.reshape(-1)

        ax.plot(x, y, 'o', label = 'RootPheno', ms = 1)
        ax.legend()
        ax.set_title(pft)

        if ratio in ['BGNPP/NPP', 'NPP/GPP']:
            ax.set_ylim([0, 1.5])
    fig.savefig(os.path.join(outdir, '..', 
                             f'factorial_limitation_compare_productivity_pft_{top}_{base}.png'),
                dpi = 600., bbox_inches = 'tight')
    plt.close(fig)

/tmp/ipykernel_2038900/1647508874.py:16: RuntimeWarning: invalid value encountered in divide
  y = rootpheno_pft.loc[:, (top, str(pft), slice(None), loc)].values.reshape(-1) / \


In [19]:
loc = 'avg'
for base in ['TOTVEGC', 'TLAI']:
    fig, axes = plt.subplots(2, 2, figsize = (10, 10), sharex = True, sharey = True)
    for i, pft in enumerate([2, 3, 11, 12]):
        ax = axes.flat[i]

        x = default.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
        y = default_pft.loc[:, (base, str(pft), slice(None), loc)].values.reshape(-1)
        ax.plot(x, y, 'o', label = 'Default', ms = 1)

        x = rootpheno.loc[:, ('TBOT', slice(None), loc)].values.reshape(-1)
        y = rootpheno_pft.loc[:, (base, str(pft), slice(None), loc)].values.reshape(-1)

        ax.plot(x, y, 'o', label = 'RootPheno', ms = 1)
        ax.legend()
        ax.set_title(pft)
    fig.savefig(os.path.join(outdir, '..', 
                             f'factorial_limitation_compare_pft_{base}.png'),
                dpi = 600., bbox_inches = 'tight')
    plt.close(fig)

## Plot the seasonality of SMINN and SMINP

In [20]:
# Compare N & P limitation of two simulations
outdir = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract')

default = pd.read_csv(os.path.join(outdir, '20231112', 'factorial_limitation_grid.csv'), 
                      index_col = 0, parse_dates = True, header = [0,1,2])
rootpheno = pd.read_csv(os.path.join(outdir, '20240304', 'factorial_limitation_grid.csv'), 
                      index_col = 0, parse_dates = True, header = [0,1,2])

cmap_cold = get_cmap('Blues')
cmap_warm = get_cmap('Reds')


loc = 'avg'
fig, axes = plt.subplots(2, 2, figsize = (13, 10), sharex = True, sharey = False)
for i, var in enumerate(['SMINN', 'SMINP']):

    ax = axes[i, 0]
    temp1 = default.loc[:, (var, slice(None), 'avg')]
    temp1.columns = temp1.columns.get_level_values(1)
    temp1 = temp1.groupby(temp1.index.month).mean()

    hcollect = {}
    for c in temp1.columns:
        co2 = chamber_levels_complete[c][1]
        tlevel = chamber_levels_complete[c][0]
        if co2 == 0:
            # ambient CO2
            color = cmap_cold((tlevel + 1) / 11)
        else:
            # elevated CO2
            color = cmap_warm((tlevel + 1) / 11)
        h, = ax.plot(np.arange(temp1.shape[0]) + 1, temp1[c], color = color)

        if (co2 == 0) & (tlevel == 0):
            hcollect.update({'Amb 0$^o$C': h})
        if (co2 == 500) & (tlevel == 0):
            hcollect.update({'Elev 0$^o$C': h})
        if (co2 == 0) & (tlevel == 9):
            hcollect.update({'Amb 9$^o$C': h})
        if (co2 == 500) & (tlevel == 9):
            hcollect.update({'Elev 9$^o$C': h})

    ax.set_title(f'{var} Default')

    ax = axes[i, 1]
    temp2 = rootpheno.loc[:, (var, slice(None), 'avg')]
    temp2.columns = temp2.columns.get_level_values(1)
    temp2 = temp2.groupby(temp2.index.month).mean()

    for c in temp2.columns:
        co2 = chamber_levels_complete[c][1]
        tlevel = chamber_levels_complete[c][0]
        if co2 == 0:
            # ambient CO2
            color = cmap_cold((tlevel + 1) / 11)
        else:
            # elevated CO2
            color = cmap_warm((tlevel + 1) / 11)
        ax.plot(np.arange(temp2.shape[0]) + 1, temp2[c], color = color)

    ax.set_title(f'{var} RootPheno')
    ax.set_xlabel('Month')

labels_handles = [(a,b) for a,b in hcollect.items()]
handles = [a[1] for a in labels_handles]
labels = [a[0] for a in labels_handles]
ax.legend(handles, labels, loc = [-0.7, -0.2], ncol = 4)
fig.savefig(os.path.join(outdir, '..', f'factorial_limitation_compare_seasonality.png'), 
            dpi = 600., bbox_inches = 'tight')
plt.close(fig)

/tmp/ipykernel_2038900/2424935369.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap_cold = get_cmap('Blues')
/tmp/ipykernel_2038900/2424935369.py:10: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap_warm = get_cmap('Reds')
